In [3]:
import os

xml_path = "/content/jaad_repo/annotations/video_0135.xml"
print("JAAD XML exists:", os.path.exists(xml_path))

JAAD XML exists: True


In [4]:
!pip install ultralytics gdown -q
!git clone https://github.com/ykotseruba/JAAD.git /content/jaad_repo
!gdown --id 1HCFLBO9fJutCKG11FtjKfdLvME6Qe_5L -O /content/JAAD_clips.zip
!mkdir -p /content/JAAD_clips
!unzip -j /content/JAAD_clips.zip "JAAD_clips/video_0135.mp4" -d /content/JAAD_clips/

fatal: destination path '/content/jaad_repo' already exists and is not an empty directory.
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1HCFLBO9fJutCKG11FtjKfdLvME6Qe_5L
From (redirected): https://drive.google.com/uc?id=1HCFLBO9fJutCKG11FtjKfdLvME6Qe_5L&confirm=t&uuid=37ccf8db-82e5-43ae-96dd-714832f187cb
To: /content/JAAD_clips.zip
100% 3.08G/3.08G [00:52<00:00, 59.0MB/s]
Archive:  /content/JAAD_clips.zip
  inflating: /content/JAAD_clips/video_0135.mp4  


In [5]:
import os

video_path = "/content/JAAD_clips/video_0135.mp4"
xml_path = "/content/jaad_repo/annotations/video_0135.xml"

print("Video exists:", os.path.exists(video_path))
print("XML exists:", os.path.exists(xml_path))

Video exists: True
XML exists: True


In [6]:
import cv2
import os
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

model = YOLO("yolov8m.pt")

video_path = "/content/JAAD_clips/video_0135.mp4"
output_folder = "/content/JAAD_visuals_yolo_conf030/"
os.makedirs(output_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)

frame_count = 0
shown_count = 0
saved_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % 30 == 0:
        results = model(
            frame,
            classes=[0],
            conf=0.30,
            iou=0.5,
            verbose=False
        )

        annotated = results[0].plot()

        save_path = f"{output_folder}jaad_yolo_conf030_frame_{frame_count:04d}.jpg"
        cv2.imwrite(save_path, annotated)

        print(f"Frame {frame_count}: {len(results[0].boxes)} people detected")

        if shown_count < 8:
            cv2_imshow(annotated)
            shown_count += 1

        saved_count += 1

    frame_count += 1

cap.release()

print("Done.")
print("Saved frames:", saved_count)
print("Folder:", output_folder)

Output hidden; open in https://colab.research.google.com to view.

In [7]:
!pip install ultralytics -q

import cv2
import os
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from ultralytics import SAM
from google.colab.patches import cv2_imshow

# =============================
# PATHS
# =============================
video_path = "/content/JAAD_clips/video_0135.mp4"
xml_path = "/content/jaad_repo/annotations/video_0135.xml"

output_folder = "/content/JAAD_sam2_only_annotation_prompts/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# SETTINGS
# =============================
frame_stride = 30
max_boxes_per_frame = 5
show_first_n = 8

# =============================
# LOAD SAM2
# =============================
sam_model = SAM("sam2.1_s.pt")

# =============================
# HELPER FUNCTIONS
# =============================
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]

# =============================
# LOAD JAAD ANNOTATIONS
# =============================
tree = ET.parse(xml_path)
root = tree.getroot()

annotations_by_frame = {}

for track in root.findall("track"):
    label = track.attrib.get("label", "")

    if label not in ["pedestrian", "ped"]:
        continue

    for box in track.findall("box"):
        if box.attrib.get("outside", "0") != "0":
            continue

        frame_id = int(box.attrib["frame"])

        x1 = float(box.attrib["xtl"])
        y1 = float(box.attrib["ytl"])
        x2 = float(box.attrib["xbr"])
        y2 = float(box.attrib["ybr"])

        box_w = x2 - x1
        box_h = y2 - y1
        area = box_w * box_h

        # keep only clearer pedestrians
        if box_h < 45:
            continue
        if area < 1000:
            continue

        annotations_by_frame.setdefault(frame_id, []).append([x1, y1, x2, y2])

print("Loaded JAAD annotation frames:", len(annotations_by_frame))

# =============================
# PROCESS VIDEO
# =============================
cap = cv2.VideoCapture(video_path)

frame_count = 0
shown_count = 0
saved_count = 0

rows = []

print("Running SAM2 only on JAAD using annotation prompts...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % frame_stride == 0:

        boxes = annotations_by_frame.get(frame_count, [])

        # keep only biggest pedestrians for cleaner output
        boxes = sorted(
            boxes,
            key=lambda b: (b[2] - b[0]) * (b[3] - b[1]),
            reverse=True
        )[:max_boxes_per_frame]

        if len(boxes) > 0:
            temp_path = "/content/temp_jaad_sam2_only.jpg"
            cv2.imwrite(temp_path, frame)

            boxes_np = np.array(boxes)

            sam_results = sam_model(
                temp_path,
                bboxes=boxes_np,
                verbose=False
            )

            if sam_results[0].masks is not None:
                masks = sam_results[0].masks.data.cpu().numpy()
            else:
                masks = []

            print(f"Frame {frame_count}: prompts={len(boxes)}, masks={len(masks)}")

            for i, mask in enumerate(masks):
                if i >= len(boxes):
                    break

                prompt_box = boxes[i]
                prompt_area = (prompt_box[2] - prompt_box[0]) * (prompt_box[3] - prompt_box[1])

                mask_bool = mask.astype(bool)
                mask_area = int(mask_bool.sum())

                mask_bbox = mask_to_bbox(mask_bool)

                if mask_bbox is not None:
                    mask_bbox_iou = compute_iou(mask_bbox, prompt_box)
                else:
                    mask_bbox_iou = 0

                mask_to_box_ratio = mask_area / prompt_area if prompt_area > 0 else 0

                rows.append({
                    "frame": frame_count,
                    "prompt_id": i + 1,
                    "prompt_box_area": round(prompt_area, 2),
                    "mask_area": mask_area,
                    "mask_to_box_ratio": round(mask_to_box_ratio, 3),
                    "mask_bbox_iou_with_prompt_box": round(mask_bbox_iou, 3)
                })

            # save visual result
            save_path = f"{output_folder}jaad_sam2_only_frame_{frame_count:04d}.jpg"
            sam_results[0].save(filename=save_path)
            saved_count += 1

            # show first few only
            if shown_count < show_first_n:
                result_img = cv2.imread(save_path)
                cv2_imshow(result_img)
                shown_count += 1

        else:
            print(f"Frame {frame_count}: no usable annotation boxes")

    frame_count += 1

cap.release()

df_jaad_sam2_only = pd.DataFrame(rows)

print("\nDone.")
print("Saved frames:", saved_count)
print("Output folder:", output_folder)
print("Total prompt-mask pairs:", len(df_jaad_sam2_only))

df_jaad_sam2_only.head()

Output hidden; open in https://colab.research.google.com to view.

In [8]:
!pip install ultralytics -q

import cv2
import os
import numpy as np
from ultralytics import YOLO, SAM
from google.colab.patches import cv2_imshow

# =============================
# PATHS
# =============================
video_path = "/content/JAAD_clips/video_0135.mp4"

output_folder = "/content/JAAD_yolo_sam2_combined_conf030/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# MODELS
# =============================
yolo_model = YOLO("yolov8m.pt")
sam_model = SAM("sam2.1_s.pt")

# =============================
# SETTINGS
# =============================
conf_threshold = 0.30
iou_threshold = 0.50
frame_stride = 30
show_first_n = 8

# =============================
# PROCESS VIDEO
# =============================
cap = cv2.VideoCapture(video_path)

frame_count = 0
saved_count = 0
shown_count = 0

print("Running JAAD combined YOLOv8m + SAM2...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % frame_stride == 0:

        # Save current frame temporarily
        temp_path = "/content/temp_jaad_combined.jpg"
        cv2.imwrite(temp_path, frame)

        # -----------------------------
        # Step 1: YOLO detection
        # -----------------------------
        yolo_results = yolo_model(
            temp_path,
            classes=[0],
            conf=conf_threshold,
            iou=iou_threshold,
            imgsz=1280,
            verbose=False
        )

        boxes = yolo_results[0].boxes.xyxy.cpu().numpy() if yolo_results[0].boxes is not None else []
        confs = yolo_results[0].boxes.conf.cpu().numpy() if yolo_results[0].boxes is not None else []

        print(f"Frame {frame_count}: {len(boxes)} people detected")

        # -----------------------------
        # Step 2: SAM2 segmentation
        # -----------------------------
        if len(boxes) > 0:
            sam_results = sam_model(
                temp_path,
                bboxes=np.array(boxes),
                verbose=False
            )

            # Start from original frame
            output = frame.copy()

            if sam_results[0].masks is not None:
                masks = sam_results[0].masks.data.cpu().numpy()
            else:
                masks = []

            colors = [
                (50, 255, 50),
                (255, 50, 50),
                (50, 50, 255),
                (255, 255, 50),
                (255, 50, 255),
                (50, 255, 255),
                (255, 165, 50),
                (150, 50, 255),
            ]

            for i, box in enumerate(boxes):
                x1, y1, x2, y2 = box.astype(int)
                color = colors[i % len(colors)]

                # draw SAM2 mask
                if i < len(masks):
                    mask_bool = masks[i].astype(bool)
                    colored = output.copy()
                    colored[mask_bool] = color
                    output = cv2.addWeighted(output, 0.7, colored, 0.3, 0)

                # draw YOLO box
                cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

                # label
                conf_text = f"{confs[i]:.2f}" if i < len(confs) else ""
                label = f"P{i+1} {conf_text}"

                cv2.putText(
                    output,
                    label,
                    (x1, max(y1 - 8, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2
                )

            # Save combined frame
            save_path = f"{output_folder}jaad_yolo_sam2_frame_{frame_count:04d}.jpg"
            cv2.imwrite(save_path, output)
            saved_count += 1

            # Show first few
            if shown_count < show_first_n:
                cv2_imshow(output)
                shown_count += 1

    frame_count += 1

cap.release()

print("\nDone.")
print("Saved frames:", saved_count)
print("Output folder:", output_folder)

Output hidden; open in https://colab.research.google.com to view.

In [9]:
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

start_frame = 6870
num_frames_to_process = 622
conf_threshold = 0.50

In [11]:
import cv2
import os
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"

start_frame = 6870
num_frames_to_process = 622
conf_threshold = 0.50

frame_stride = 30
show_first_n = 25   # show up to 25 pictures directly in Colab

output_folder = "/content/PIE_video_0006_yolo_only_conf050/"
os.makedirs(output_folder, exist_ok=True)

model = YOLO("yolov8m.pt")

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

local_frame_idx = 0
shown_count = 0
saved_count = 0

print("Showing YOLOv8m-only PIE results in Colab...")

while local_frame_idx < num_frames_to_process:
    ret, frame = cap.read()
    if not ret:
        break

    original_frame = start_frame + local_frame_idx

    if local_frame_idx % frame_stride == 0:
        results = model(
            frame,
            classes=[0],
            conf=conf_threshold,
            iou=0.5,
            verbose=False
        )

        annotated = results[0].plot()

        save_path = f"{output_folder}pie_yolo_frame_{original_frame}.jpg"
        cv2.imwrite(save_path, annotated)
        saved_count += 1

        print(f"\nFrame {original_frame}: {len(results[0].boxes)} people detected")

        if shown_count < show_first_n:
            cv2_imshow(annotated)
            shown_count += 1

    local_frame_idx += 1

cap.release()

print("\nDone.")
print("Shown images:", shown_count)
print("Saved images:", saved_count)
print("Output folder:", output_folder)

Showing YOLOv8m-only PIE results in Colab...

Done.
Shown images: 0
Saved images: 0
Output folder: /content/PIE_video_0006_yolo_only_conf050/


In [12]:
!git clone https://github.com/aras62/PIE.git /content/PIE

Cloning into '/content/PIE'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 178 (delta 32), reused 75 (delta 17), pack-reused 85 (from 1)
Receiving objects: 100% (178/178), 144.63 MiB | 25.76 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Updating files: 100% (41/41), done.


In [13]:
import os

os.makedirs("/content/PIE/annotations_unzipped", exist_ok=True)

!unzip -q /content/PIE/annotations/annotations.zip -d /content/PIE/annotations_unzipped

In [14]:
import os

xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

print("XML exists:", os.path.exists(xml_path))
print(xml_path)

XML exists: True
/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml


In [15]:
import os

os.makedirs("/content/PIE/annotations/PIE_clips/set03", exist_ok=True)

!wget -O /content/PIE/annotations/PIE_clips/set03/video_0006.mp4 \
https://data.nvision2.eecs.yorku.ca/PIE_dataset/PIE_clips/set03/video_0006.mp4

--2026-05-13 17:27:22--  https://data.nvision2.eecs.yorku.ca/PIE_dataset/PIE_clips/set03/video_0006.mp4
Resolving data.nvision2.eecs.yorku.ca (data.nvision2.eecs.yorku.ca)... 130.63.94.247
Connecting to data.nvision2.eecs.yorku.ca (data.nvision2.eecs.yorku.ca)|130.63.94.247|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1500436832 (1.4G) [video/mp4]
Saving to: ‘/content/PIE/annotations/PIE_clips/set03/video_0006.mp4’

/content/PIE/annota 100%[===================>]   1.40G  68.0MB/s    in 26s     

2026-05-13 17:27:49 (54.2 MB/s) - ‘/content/PIE/annotations/PIE_clips/set03/video_0006.mp4’ saved [1500436832/1500436832]



In [16]:
import os
import cv2

video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

print("Video exists:", os.path.exists(video_path))
print("XML exists:", os.path.exists(xml_path))

cap = cv2.VideoCapture(video_path)
print("Video opened:", cap.isOpened())
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Total frames:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

Video exists: True
XML exists: True
Video opened: True
FPS: 30.0
Total frames: 18000.0


In [17]:
import cv2
import os
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"

start_frame = 6870
num_frames_to_process = 622
conf_threshold = 0.50

frame_stride = 30
show_first_n = 25

output_folder = "/content/PIE_video_0006_yolo_only_conf050/"
os.makedirs(output_folder, exist_ok=True)

model = YOLO("yolov8m.pt")

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

local_frame_idx = 0
shown_count = 0
saved_count = 0

print("Showing YOLOv8m-only PIE results in Colab...")

while local_frame_idx < num_frames_to_process:
    ret, frame = cap.read()
    if not ret:
        print("Could not read frame at local index:", local_frame_idx)
        break

    original_frame = start_frame + local_frame_idx

    if local_frame_idx % frame_stride == 0:
        results = model(
            frame,
            classes=[0],
            conf=conf_threshold,
            iou=0.5,
            verbose=False
        )

        annotated = results[0].plot()

        save_path = f"{output_folder}pie_yolo_frame_{original_frame}.jpg"
        cv2.imwrite(save_path, annotated)
        saved_count += 1

        print(f"\nFrame {original_frame}: {len(results[0].boxes)} people detected")

        if shown_count < show_first_n:
            cv2_imshow(annotated)
            shown_count += 1

    local_frame_idx += 1

cap.release()

print("\nDone.")
print("Shown images:", shown_count)
print("Saved images:", saved_count)
print("Output folder:", output_folder)

Output hidden; open in https://colab.research.google.com to view.

In [18]:
import cv2
import os
import numpy as np
import xml.etree.ElementTree as ET
from ultralytics import SAM
from google.colab.patches import cv2_imshow

# =============================
# PATHS
# =============================
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

output_folder = "/content/PIE_video_0006_sam2_only_annotation_prompts/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# SETTINGS
# =============================
start_frame = 6870
num_frames_to_process = 622
frame_stride = 30
show_first_n = 25

# Keep more people visible
max_boxes_per_frame = 20

# =============================
# LOAD SAM2
# =============================
sam_model = SAM("sam2.1_s.pt")

# =============================
# LOAD PIE ANNOTATION BOXES
# =============================
tree = ET.parse(xml_path)
root = tree.getroot()

annotations_by_frame = {}

for track in root.findall("track"):
    if track.attrib.get("label", "") != "pedestrian":
        continue

    for box in track.findall("box"):
        if box.attrib.get("outside", "0") == "1":
            continue

        frame_id = int(box.attrib["frame"])

        x1 = float(box.attrib["xtl"])
        y1 = float(box.attrib["ytl"])
        x2 = float(box.attrib["xbr"])
        y2 = float(box.attrib["ybr"])

        box_w = x2 - x1
        box_h = y2 - y1
        area = box_w * box_h

        # Less strict filtering, so we see more people
        if box_h < 25:
            continue
        if area < 300:
            continue

        annotations_by_frame.setdefault(frame_id, []).append([x1, y1, x2, y2])

print("Loaded annotation frames:", len(annotations_by_frame))

# =============================
# PROCESS VIDEO SEGMENT
# =============================
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

local_frame_idx = 0
shown_count = 0
saved_count = 0

print("Running SAM2 only using PIE annotation boxes as prompts...")

while local_frame_idx < num_frames_to_process:
    ret, frame = cap.read()
    if not ret:
        print("Could not read frame at local index:", local_frame_idx)
        break

    original_frame = start_frame + local_frame_idx

    if local_frame_idx % frame_stride == 0:

        boxes = annotations_by_frame.get(original_frame, [])

        # Keep biggest boxes first, but allow many
        boxes = sorted(
            boxes,
            key=lambda b: (b[2] - b[0]) * (b[3] - b[1]),
            reverse=True
        )[:max_boxes_per_frame]

        print(f"\nFrame {original_frame}: annotation prompts = {len(boxes)}")

        if len(boxes) > 0:
            temp_path = "/content/temp_pie_sam2_only.jpg"
            cv2.imwrite(temp_path, frame)

            sam_results = sam_model(
                temp_path,
                bboxes=np.array(boxes),
                verbose=False
            )

            save_path = f"{output_folder}pie_sam2_only_frame_{original_frame}.jpg"
            sam_results[0].save(filename=save_path)
            saved_count += 1

            result_img = cv2.imread(save_path)

            if shown_count < show_first_n:
                cv2_imshow(result_img)
                shown_count += 1

        else:
            print("No usable annotation boxes for this frame.")

    local_frame_idx += 1

cap.release()

print("\nDone.")
print("Shown images:", shown_count)
print("Saved images:", saved_count)
print("Output folder:", output_folder)

Output hidden; open in https://colab.research.google.com to view.

In [19]:
import cv2
import os
import numpy as np
from ultralytics import YOLO, SAM
from google.colab.patches import cv2_imshow

# =============================
# PATHS
# =============================
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"

output_folder = "/content/PIE_video_0006_yolo_sam2_combined_conf050/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# SETTINGS
# =============================
start_frame = 6870
num_frames_to_process = 622
frame_stride = 30
show_first_n = 25

# Best PIE confidence
conf_threshold = 0.50
iou_threshold = 0.50

# =============================
# LOAD MODELS
# =============================
yolo_model = YOLO("yolov8m.pt")
sam_model = SAM("sam2.1_s.pt")

# =============================
# PROCESS VIDEO SEGMENT
# =============================
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

local_frame_idx = 0
shown_count = 0
saved_count = 0

print("Running PIE combined YOLOv8m + SAM2 at confidence 0.50...")

while local_frame_idx < num_frames_to_process:
    ret, frame = cap.read()
    if not ret:
        print("Could not read frame at local index:", local_frame_idx)
        break

    original_frame = start_frame + local_frame_idx

    if local_frame_idx % frame_stride == 0:

        temp_path = "/content/temp_pie_combined.jpg"
        cv2.imwrite(temp_path, frame)

        # -----------------------------
        # Step 1: YOLO detects people
        # -----------------------------
        yolo_results = yolo_model(
            temp_path,
            classes=[0],
            conf=conf_threshold,
            iou=iou_threshold,
            imgsz=1280,
            verbose=False
        )

        boxes = yolo_results[0].boxes.xyxy.cpu().numpy() if yolo_results[0].boxes is not None else []

        print(f"\nFrame {original_frame}: YOLO detections = {len(boxes)}")

        # -----------------------------
        # Step 2: SAM2 segments YOLO boxes
        # -----------------------------
        if len(boxes) > 0:
            sam_results = sam_model(
                temp_path,
                bboxes=np.array(boxes),
                verbose=False
            )

            save_path = f"{output_folder}pie_yolo_sam2_combined_frame_{original_frame}.jpg"
            sam_results[0].save(filename=save_path)
            saved_count += 1

            result_img = cv2.imread(save_path)

            if shown_count < show_first_n:
                cv2_imshow(result_img)
                shown_count += 1

        else:
            print("No YOLO detections, so SAM2 was not applied.")

    local_frame_idx += 1

cap.release()

print("\nDone.")
print("Shown images:", shown_count)
print("Saved images:", saved_count)
print("Output folder:", output_folder)

Output hidden; open in https://colab.research.google.com to view.